## Import Library dan Direktori Project

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import numpy as np
import json
import re
import sys
import importlib
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

direktori_aktif = Path.cwd().resolve()

if direktori_aktif.name == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_data = direktori_project / "data"
direktori_raw = direktori_data / "raw"
direktori_processed = direktori_data / "processed"
direktori_intelligence = direktori_data / "intelligence"
direktori_corrections = direktori_data / "corrections"
direktori_models = direktori_project / "models"
direktori_reports = direktori_project / "reports"
direktori_outputs = direktori_reports / "outputs"
direktori_src = direktori_project / "src"

for folder in [
    direktori_processed,
    direktori_models,
    direktori_outputs
]:
    folder.mkdir(parents=True, exist_ok=True)

if str(direktori_src) not in sys.path:
    sys.path.append(str(direktori_src))

print("Direktori aktif notebook:", direktori_aktif)
print("Direktori project:", direktori_project)
print("Folder processed:", direktori_processed)
print("Folder models:", direktori_models)
print("Folder outputs:", direktori_outputs)

Direktori aktif notebook: C:\Users\ASUS\PHISHING\notebooks
Direktori project: C:\Users\ASUS\PHISHING
Folder processed: C:\Users\ASUS\PHISHING\data\processed
Folder models: C:\Users\ASUS\PHISHING\models
Folder outputs: C:\Users\ASUS\PHISHING\reports\outputs


## Load File Penting

In [2]:
lokasi_dataset = direktori_raw / "PhiUSIIL_Phishing_URL_Dataset.csv"
lokasi_fitur_url_manual = direktori_outputs / "daftar_fitur_url_manual.json"

lokasi_domain_resmi = direktori_intelligence / "official_domains_global.csv"
lokasi_brand_keyword = direktori_intelligence / "brand_keywords_global.csv"
lokasi_suspicious_keyword = direktori_intelligence / "suspicious_keywords_global.csv"
lokasi_url_sintetis = direktori_intelligence / "generated_suspicious_urls.csv"
lokasi_url_corrections = direktori_corrections / "url_corrections.csv"

daftar_file_wajib = [
    lokasi_dataset,
    lokasi_fitur_url_manual,
    lokasi_domain_resmi,
    lokasi_brand_keyword,
    lokasi_suspicious_keyword,
    lokasi_url_sintetis,
    direktori_src / "url_intelligence.py"
]

for lokasi_file in daftar_file_wajib:
    if not lokasi_file.exists():
        raise FileNotFoundError(f"File wajib tidak ditemukan: {lokasi_file}")

data_asli = pd.read_csv(lokasi_dataset)

with open(lokasi_fitur_url_manual, "r", encoding="utf-8") as file:
    daftar_fitur_url_manual = json.load(file)

data_domain_resmi_global = pd.read_csv(lokasi_domain_resmi)
data_brand_keyword_global = pd.read_csv(lokasi_brand_keyword)
data_suspicious_keyword_global = pd.read_csv(lokasi_suspicious_keyword)
data_url_sintetis = pd.read_csv(lokasi_url_sintetis)

print("Dataset asli berhasil dibaca.")
print("Ukuran dataset:", data_asli.shape)
print("Jumlah fitur URL manual:", len(daftar_fitur_url_manual))
print("Jumlah domain resmi:", len(data_domain_resmi_global))
print("Jumlah brand keyword:", len(data_brand_keyword_global))
print("Jumlah suspicious keyword:", len(data_suspicious_keyword_global))
print("Jumlah URL sintetis:", len(data_url_sintetis))

Dataset asli berhasil dibaca.
Ukuran dataset: (235795, 56)
Jumlah fitur URL manual: 39
Jumlah domain resmi: 50
Jumlah brand keyword: 45
Jumlah suspicious keyword: 30
Jumlah URL sintetis: 242


## Membuat Fungsi Fitur URL Manual

In [3]:
def ambil_domain_dari_url(url):
    url = str(url).strip()
    hasil_parse = urlparse(url)

    if hasil_parse.netloc:
        return hasil_parse.netloc.lower().split("@")[-1].split(":")[0]

    hasil_parse = urlparse("http://" + url)
    return hasil_parse.netloc.lower().split("@")[-1].split(":")[0]


def cek_domain_ip(domain):
    pola_ip = r"^\d{1,3}(\.\d{1,3}){3}$"
    return int(bool(re.match(pola_ip, str(domain).strip())))


def ambil_tld(domain):
    bagian = [item for item in str(domain).lower().split(".") if item]

    if not bagian:
        return "tidak_diketahui"

    return bagian[-1]


def hitung_subdomain(domain):
    domain = str(domain).lower().strip()
    bagian = [item for item in domain.split(".") if item]

    if cek_domain_ip(domain):
        return 0

    if len(bagian) <= 2:
        return 0

    return len(bagian) - 2


def hapus_skema_url(url):
    return re.sub(r"^https?://", "", str(url).strip(), flags=re.IGNORECASE)


def hapus_www_awal(teks):
    return re.sub(r"^www\.", "", str(teks).strip(), flags=re.IGNORECASE)


def hitung_obfuscation(url):
    pola = r"%[0-9a-fA-F]{2}"
    return len(re.findall(pola, str(url)))


def ekstrak_fitur_url_manual_satu(url, daftar_fitur):
    url = str(url).strip()
    domain = ambil_domain_dari_url(url)
    tld = ambil_tld(domain)

    url_tanpa_skema = hapus_skema_url(url)
    url_untuk_hitung = hapus_www_awal(url_tanpa_skema)

    panjang_url = len(url)
    panjang_domain = len(domain)

    jumlah_obfuscation = hitung_obfuscation(url)
    jumlah_huruf = sum(karakter.isalpha() for karakter in url_untuk_hitung)
    jumlah_angka = sum(karakter.isdigit() for karakter in url_untuk_hitung)

    jumlah_sama_dengan = url_untuk_hitung.count("=")
    jumlah_tanda_tanya = url_untuk_hitung.count("?")
    jumlah_ampersand = url_untuk_hitung.count("&")

    karakter_khusus = re.findall(r"[^a-zA-Z0-9]", url_untuk_hitung)
    jumlah_karakter_khusus = len(karakter_khusus)

    fitur = {
        "URLLength": panjang_url,
        "DomainLength": panjang_domain,
        "IsDomainIP": cek_domain_ip(domain),
        "TLDLength": len(tld),
        "NoOfSubDomain": hitung_subdomain(domain),
        "HasObfuscation": int(jumlah_obfuscation > 0),
        "NoOfObfuscatedChar": jumlah_obfuscation * 3,
        "ObfuscationRatio": (jumlah_obfuscation * 3 / panjang_url) if panjang_url > 0 else 0,
        "NoOfLettersInURL": jumlah_huruf,
        "LetterRatioInURL": (jumlah_huruf / panjang_url) if panjang_url > 0 else 0,
        "NoOfDegitsInURL": jumlah_angka,
        "DegitRatioInURL": (jumlah_angka / panjang_url) if panjang_url > 0 else 0,
        "NoOfEqualsInURL": jumlah_sama_dengan,
        "NoOfQMarkInURL": jumlah_tanda_tanya,
        "NoOfAmpersandInURL": jumlah_ampersand,
        "NoOfOtherSpecialCharsInURL": jumlah_karakter_khusus,
        "SpacialCharRatioInURL": (jumlah_karakter_khusus / panjang_url) if panjang_url > 0 else 0,
        "IsHTTPS": int(url.lower().startswith("https://"))
    }

    for nama_fitur in daftar_fitur:
        if nama_fitur.startswith("TLD_"):
            nama_tld = nama_fitur.replace("TLD_", "")
            fitur[nama_fitur] = int(tld == nama_tld)

    if f"TLD_{tld}" not in daftar_fitur and "TLD_lainnya" in daftar_fitur:
        fitur["TLD_lainnya"] = 1

    return fitur


def buat_fitur_manual_dari_daftar_url(daftar_url, daftar_fitur):
    hasil = []

    for url in daftar_url:
        hasil.append(ekstrak_fitur_url_manual_satu(url, daftar_fitur))

    data_fitur = pd.DataFrame(hasil)
    data_fitur = data_fitur.reindex(columns=daftar_fitur, fill_value=0)

    return data_fitur


def buat_fitur_manual_dari_dataset(data, daftar_fitur):
    data_fitur = pd.DataFrame(index=data.index)

    fitur_tld = [fitur for fitur in daftar_fitur if fitur.startswith("TLD_")]
    fitur_non_tld = [fitur for fitur in daftar_fitur if not fitur.startswith("TLD_")]

    for fitur in fitur_non_tld:
        if fitur in data.columns:
            data_fitur[fitur] = data[fitur]
        else:
            data_fitur[fitur] = 0

    data_tld = data["TLD"].astype(str).str.lower().str.strip()

    daftar_tld_model = [
        fitur.replace("TLD_", "")
        for fitur in fitur_tld
        if fitur != "TLD_lainnya"
    ]

    for fitur in fitur_tld:
        nama_tld = fitur.replace("TLD_", "")

        if fitur == "TLD_lainnya":
            data_fitur[fitur] = (~data_tld.isin(daftar_tld_model)).astype(int)
        else:
            data_fitur[fitur] = (data_tld == nama_tld).astype(int)

    data_fitur = data_fitur.reindex(columns=daftar_fitur, fill_value=0)

    return data_fitur

## Membuat Fitur Intelligence Cepat untuk Dataset Asli

In [4]:
FITUR_INTELLIGENCE_V2 = [
    "is_official_domain",
    "brand_keyword_detected",
    "brand_but_not_official",
    "suspicious_keyword_count",
    "suspicious_keyword_score",
    "lookalike_brand_detected",
    "lookalike_score",
    "uses_punycode",
    "uses_digit_substitution",
    "hyphen_count"
]


def bersihkan_domain_ringkas(domain):
    domain = str(domain).lower().strip()
    domain = domain.replace("http://", "").replace("https://", "")
    domain = domain.split("/")[0]
    domain = domain.split("@")[-1]
    domain = domain.split(":")[0]

    if domain.startswith("www."):
        domain = domain[4:]

    return domain


def buat_fitur_intelligence_cepat(data, data_domain_resmi, data_brand_keyword, data_suspicious_keyword):
    data_hasil = pd.DataFrame(index=data.index)

    url_lower = data["URL"].astype(str).str.lower()

    if "Domain" in data.columns:
        domain_lower = data["Domain"].astype(str).str.lower().str.strip()
    else:
        domain_lower = data["URL"].astype(str).apply(ambil_domain_dari_url)

    domain_bersih = domain_lower.apply(bersihkan_domain_ringkas)

    data_hasil["is_official_domain"] = 0

    for domain_resmi in data_domain_resmi["domain"].astype(str).str.lower().str.strip().tolist():
        domain_resmi_bersih = bersihkan_domain_ringkas(domain_resmi)
        cocok = (domain_bersih == domain_resmi_bersih) | (domain_bersih.str.endswith("." + domain_resmi_bersih))
        data_hasil.loc[cocok, "is_official_domain"] = 1

    data_hasil["brand_keyword_detected"] = 0

    for keyword in data_brand_keyword["keyword"].astype(str).str.lower().str.strip().tolist():
        if len(keyword) <= 2:
            pola = r"(^|[^a-z0-9])" + re.escape(keyword) + r"([^a-z0-9]|$)"
            cocok = url_lower.str.contains(pola, regex=True, na=False)
        else:
            cocok = url_lower.str.contains(re.escape(keyword), regex=True, na=False)

        data_hasil.loc[cocok, "brand_keyword_detected"] = 1

    data_hasil["brand_but_not_official"] = (
        (data_hasil["brand_keyword_detected"] == 1) &
        (data_hasil["is_official_domain"] == 0)
    ).astype(int)

    data_hasil["suspicious_keyword_count"] = 0
    data_hasil["suspicious_keyword_score"] = 0

    for _, baris in data_suspicious_keyword.iterrows():
        keyword = str(baris["keyword"]).lower().strip()
        bobot = int(baris["bobot"])

        pola = r"(^|[^a-z0-9])" + re.escape(keyword) + r"([^a-z0-9]|$)"
        cocok = url_lower.str.contains(pola, regex=True, na=False)

        data_hasil.loc[cocok, "suspicious_keyword_count"] += 1
        data_hasil.loc[cocok, "suspicious_keyword_score"] += bobot

    data_hasil["lookalike_brand_detected"] = 0
    data_hasil["lookalike_score"] = 0.0
    data_hasil["uses_punycode"] = domain_bersih.str.contains("xn--", regex=False, na=False).astype(int)
    data_hasil["uses_digit_substitution"] = domain_bersih.str.contains(r"[0134578]", regex=True, na=False).astype(int)
    data_hasil["hyphen_count"] = domain_bersih.str.count("-").astype(int)

    data_hasil = data_hasil.reindex(columns=FITUR_INTELLIGENCE_V2, fill_value=0)

    return data_hasil


X_manual_asli = buat_fitur_manual_dari_dataset(data_asli, daftar_fitur_url_manual)
X_intelligence_asli = buat_fitur_intelligence_cepat(
    data_asli,
    data_domain_resmi_global,
    data_brand_keyword_global,
    data_suspicious_keyword_global
)

y_asli = data_asli["label"].map({
    0: 1,
    1: 0
}).astype(int)

print("Fitur manual dataset asli:", X_manual_asli.shape)
print("Fitur intelligence dataset asli:", X_intelligence_asli.shape)
print("Target dataset asli:", y_asli.shape)

display(X_intelligence_asli.head())
display(y_asli.value_counts())

Fitur manual dataset asli: (235795, 39)
Fitur intelligence dataset asli: (235795, 10)
Target dataset asli: (235795,)


,is_official_domain,brand_keyword_detected,brand_but_not_official,suspicious_keyword_count,suspicious_keyword_score,lookalike_brand_detected,lookalike_score,uses_punycode,uses_digit_substitution,hyphen_count
0,0,0,0,0,0,0,0.0,0,0,0
1,0,0,0,0,0,0,0.0,0,0,1
2,0,0,0,0,0,0,0.0,0,0,0
3,0,0,0,0,0,0,0.0,0,0,0
4,0,1,1,0,0,0,0.0,0,0,0


label
0    134850
1    100945
Name: count, dtype: int64

## Membuat Data Tambahan Domain Resmi

In [5]:
data_official_seed = data_domain_resmi_global.copy()

data_url_official_seed = pd.DataFrame({
    "URL": "https://" + data_official_seed["domain"].astype(str).str.lower().str.strip(),
    "target_phishing": 0,
    "sumber_data": "official_domain_seed"
})

data_url_official_seed = data_url_official_seed.drop_duplicates(subset=["URL"]).reset_index(drop=True)

print("Jumlah URL resmi tambahan:", len(data_url_official_seed))
data_url_official_seed.head(10)

Jumlah URL resmi tambahan: 50


,URL,target_phishing,sumber_data
0,https://cimb.com,0,official_domain_seed
1,https://dbs.com.sg,0,official_domain_seed
2,https://kasikornbank.com,0,official_domain_seed
3,https://maybank2u.com.my,0,official_domain_seed
4,https://ocbc.com,0,official_domain_seed
5,https://scb.co.th,0,official_domain_seed
6,https://uob.com.sg,0,official_domain_seed
7,https://grab.com,0,official_domain_seed
8,https://amazon.com,0,official_domain_seed
9,https://netflix.com,0,official_domain_seed


## Membuat Data Tambahan URL Sintetis Mencurigakan

In [6]:
data_url_sintetis_model = data_url_sintetis.rename(columns={
    "url": "URL"
}).copy()

data_url_sintetis_model["target_phishing"] = 1
data_url_sintetis_model["sumber_data"] = "synthetic_suspicious_url"

data_url_sintetis_model = data_url_sintetis_model[[
    "URL",
    "target_phishing",
    "sumber_data"
]].drop_duplicates(subset=["URL"]).reset_index(drop=True)

print("Jumlah URL sintetis mencurigakan:", len(data_url_sintetis_model))
data_url_sintetis_model.head(10)

Jumlah URL sintetis mencurigakan: 242


,URL,target_phishing,sumber_data
0,http://cimb-login.test,1,synthetic_suspicious_url
1,http://cimb-verify.test,1,synthetic_suspicious_url
2,http://cimb-update.test,1,synthetic_suspicious_url
3,http://cimb-account.test,1,synthetic_suspicious_url
4,http://cimbb.test,1,synthetic_suspicious_url
5,http://c1mb.test,1,synthetic_suspicious_url
6,http://cimbsecure.test,1,synthetic_suspicious_url
7,http://ccimb.test,1,synthetic_suspicious_url
8,http://cimb.login-update.test,1,synthetic_suspicious_url
9,http://dbs-login.test,1,synthetic_suspicious_url


## Membaca Data Koreksi

In [7]:
if lokasi_url_corrections.exists():
    data_url_corrections = pd.read_csv(lokasi_url_corrections)

    if not data_url_corrections.empty and "url" in data_url_corrections.columns and "label_benar" in data_url_corrections.columns:
        data_url_corrections_model = data_url_corrections.rename(columns={
            "url": "URL",
            "label_benar": "target_phishing"
        }).copy()

        data_url_corrections_model["target_phishing"] = data_url_corrections_model["target_phishing"].astype(int)
        data_url_corrections_model["sumber_data"] = "user_correction"

        data_url_corrections_model = data_url_corrections_model[[
            "URL",
            "target_phishing",
            "sumber_data"
        ]].drop_duplicates(subset=["URL"]).reset_index(drop=True)
    else:
        data_url_corrections_model = pd.DataFrame(columns=["URL", "target_phishing", "sumber_data"])
else:
    data_url_corrections_model = pd.DataFrame(columns=["URL", "target_phishing", "sumber_data"])

print("Jumlah data koreksi URL:", len(data_url_corrections_model))
data_url_corrections_model.head()

Jumlah data koreksi URL: 0


,URL,target_phishing,sumber_data


## Membuat Fitur untuk Data Tambahan

In [8]:
import url_intelligence
importlib.reload(url_intelligence)

data_domain_resmi, data_brand_keyword, data_suspicious_keyword = url_intelligence.muat_data_intelligence(direktori_project)


def buat_fitur_tambahan_v2(data_url_tambahan, daftar_fitur_url_manual):
    if data_url_tambahan.empty:
        return pd.DataFrame(), pd.Series(dtype=int), pd.DataFrame()

    daftar_url = data_url_tambahan["URL"].astype(str).tolist()

    X_manual = buat_fitur_manual_dari_daftar_url(
        daftar_url,
        daftar_fitur_url_manual
    )

    data_intelligence_lengkap = url_intelligence.analisis_banyak_url(
        daftar_url,
        data_domain_resmi,
        data_brand_keyword,
        data_suspicious_keyword
    )

    X_intelligence = data_intelligence_lengkap[FITUR_INTELLIGENCE_V2].copy()
    X_intelligence = X_intelligence.fillna(0)

    X_gabungan = pd.concat(
        [
            X_manual.reset_index(drop=True),
            X_intelligence.reset_index(drop=True)
        ],
        axis=1
    )

    y = data_url_tambahan["target_phishing"].astype(int).reset_index(drop=True)

    metadata = data_url_tambahan[["URL", "sumber_data"]].reset_index(drop=True)

    return X_gabungan, y, metadata


X_official, y_official, metadata_official = buat_fitur_tambahan_v2(
    data_url_official_seed,
    daftar_fitur_url_manual
)

X_sintetis, y_sintetis, metadata_sintetis = buat_fitur_tambahan_v2(
    data_url_sintetis_model,
    daftar_fitur_url_manual
)

X_koreksi, y_koreksi, metadata_koreksi = buat_fitur_tambahan_v2(
    data_url_corrections_model,
    daftar_fitur_url_manual
)

print("X official:", X_official.shape)
print("X sintetis:", X_sintetis.shape)
print("X koreksi:", X_koreksi.shape)

X official: (50, 49)
X sintetis: (242, 49)
X koreksi: (0, 0)


## Menggabungkan Dataset Training V2

In [9]:
daftar_fitur_model_v2 = daftar_fitur_url_manual + FITUR_INTELLIGENCE_V2

X_asli_v2 = pd.concat(
    [
        X_manual_asli.reset_index(drop=True),
        X_intelligence_asli.reset_index(drop=True)
    ],
    axis=1
)

metadata_asli = pd.DataFrame({
    "URL": data_asli["URL"].astype(str).values,
    "sumber_data": "dataset_asli"
})

X_semua = pd.concat(
    [
        X_asli_v2,
        X_official,
        X_sintetis,
        X_koreksi
    ],
    ignore_index=True
)

y_semua = pd.concat(
    [
        y_asli.reset_index(drop=True),
        y_official,
        y_sintetis,
        y_koreksi
    ],
    ignore_index=True
)

metadata_semua = pd.concat(
    [
        metadata_asli,
        metadata_official,
        metadata_sintetis,
        metadata_koreksi
    ],
    ignore_index=True
)

X_semua = X_semua.reindex(columns=daftar_fitur_model_v2, fill_value=0)
X_semua = X_semua.fillna(0)

data_training_v2 = metadata_semua.copy()
data_training_v2["target_phishing"] = y_semua.values
data_training_v2 = pd.concat([data_training_v2, X_semua], axis=1)

def tentukan_bobot_data(sumber_data):
    if sumber_data == "official_domain_seed":
        return 12.0

    if sumber_data == "synthetic_suspicious_url":
        return 8.0

    if sumber_data == "user_correction":
        return 15.0

    return 1.0

data_training_v2["sample_weight"] = data_training_v2["sumber_data"].apply(tentukan_bobot_data)

print("Ukuran data training V2:", data_training_v2.shape)
print("Jumlah fitur model V2:", len(daftar_fitur_model_v2))

display(data_training_v2["sumber_data"].value_counts())
display(data_training_v2["target_phishing"].value_counts())
display(data_training_v2[["URL", "sumber_data", "target_phishing", "sample_weight"]].head())

Ukuran data training V2: (236087, 53)
Jumlah fitur model V2: 49


sumber_data
dataset_asli                235795
synthetic_suspicious_url       242
official_domain_seed            50
Name: count, dtype: int64

target_phishing
0    134900
1    101187
Name: count, dtype: int64

,URL,sumber_data,target_phishing,sample_weight
0,https://www.southbankmosaics.com,dataset_asli,0,1.0
1,https://www.uni-mainz.de,dataset_asli,0,1.0
2,https://www.voicefmradio.co.uk,dataset_asli,0,1.0
3,https://www.sfnmjournal.com,dataset_asli,0,1.0
4,https://www.rewildingargentina.org,dataset_asli,0,1.0


## Simpan Dataset Training V2

In [10]:
lokasi_training_v2 = direktori_processed / "dataset_training_intelligence_v2.csv"
lokasi_fitur_intelligence_v2 = direktori_outputs / "daftar_fitur_intelligence_v2.json"

data_training_v2.to_csv(
    lokasi_training_v2,
    index=False,
    encoding="utf-8"
)

with open(lokasi_fitur_intelligence_v2, "w", encoding="utf-8") as file:
    json.dump(daftar_fitur_model_v2, file, indent=4, ensure_ascii=False)

print("Dataset training V2 disimpan:")
print(lokasi_training_v2)

print("Daftar fitur model V2 disimpan:")
print(lokasi_fitur_intelligence_v2)

Dataset training V2 disimpan:
C:\Users\ASUS\PHISHING\data\processed\dataset_training_intelligence_v2.csv
Daftar fitur model V2 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\daftar_fitur_intelligence_v2.json


## Split Data Train dan Test

In [11]:
X = data_training_v2[daftar_fitur_model_v2]
y = data_training_v2["target_phishing"].astype(int)
sample_weight = data_training_v2["sample_weight"].astype(float)

X_train_v2, X_test_v2, y_train_v2, y_test_v2, weight_train_v2, weight_test_v2 = train_test_split(
    X,
    y,
    sample_weight,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Ukuran X_train_v2:", X_train_v2.shape)
print("Ukuran X_test_v2:", X_test_v2.shape)
print("Ukuran y_train_v2:", y_train_v2.shape)
print("Ukuran y_test_v2:", y_test_v2.shape)

print("\nDistribusi target train:")
print(y_train_v2.value_counts())

print("\nDistribusi target test:")
print(y_test_v2.value_counts())

Ukuran X_train_v2: (188869, 49)
Ukuran X_test_v2: (47218, 49)
Ukuran y_train_v2: (188869,)
Ukuran y_test_v2: (47218,)

Distribusi target train:
target_phishing
0    107920
1     80949
Name: count, dtype: int64

Distribusi target test:
target_phishing
0    26980
1    20238
Name: count, dtype: int64


## Training Model Random Forest Intelligence V2

In [12]:
model_rf_intelligence_v2 = RandomForestClassifier(
    n_estimators=250,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model_rf_intelligence_v2.fit(
    X_train_v2,
    y_train_v2,
    sample_weight=weight_train_v2
)

print("Training Random Forest Intelligence V2 selesai.")

Training Random Forest Intelligence V2 selesai.


## Training XGBoost Intelligence V2 Jika Tersedia

In [13]:
model_xgb_intelligence_v2 = None

try:
    from xgboost import XGBClassifier

    model_xgb_intelligence_v2 = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model_xgb_intelligence_v2.fit(
        X_train_v2,
        y_train_v2,
        sample_weight=weight_train_v2
    )

    print("Training XGBoost Intelligence V2 selesai.")

except Exception as error:
    print("XGBoost dilewati karena tidak tersedia atau gagal dijalankan.")
    print(error)

Training XGBoost Intelligence V2 selesai.


## Evaluasi Model Intelligence V2

In [14]:
def evaluasi_model_v2(nama_model, model, X_test, y_test):
    prediksi = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        probabilitas = model.predict_proba(X_test)[:, 1]
    else:
        probabilitas = prediksi

    hasil = {
        "nama_model": nama_model,
        "accuracy": round(accuracy_score(y_test, prediksi), 4),
        "precision_phishing": round(precision_score(y_test, prediksi, pos_label=1), 4),
        "recall_phishing": round(recall_score(y_test, prediksi, pos_label=1), 4),
        "f1_phishing": round(f1_score(y_test, prediksi, pos_label=1), 4),
        "roc_auc": round(roc_auc_score(y_test, probabilitas), 4)
    }

    print(f"Evaluasi {nama_model}")
    print(classification_report(
        y_test,
        prediksi,
        target_names=["Legitimate", "Phishing"]
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, prediksi))

    return hasil


daftar_hasil_evaluasi_v2 = []

hasil_rf_v2 = evaluasi_model_v2(
    "Random Forest Intelligence V2",
    model_rf_intelligence_v2,
    X_test_v2,
    y_test_v2
)

daftar_hasil_evaluasi_v2.append(hasil_rf_v2)

if model_xgb_intelligence_v2 is not None:
    hasil_xgb_v2 = evaluasi_model_v2(
        "XGBoost Intelligence V2",
        model_xgb_intelligence_v2,
        X_test_v2,
        y_test_v2
    )

    daftar_hasil_evaluasi_v2.append(hasil_xgb_v2)

data_hasil_evaluasi_v2 = pd.DataFrame(daftar_hasil_evaluasi_v2).sort_values(
    by=["f1_phishing", "recall_phishing", "roc_auc"],
    ascending=False
).reset_index(drop=True)

lokasi_hasil_evaluasi_v2 = direktori_outputs / "hasil_evaluasi_model_intelligence_v2.csv"

data_hasil_evaluasi_v2.to_csv(
    lokasi_hasil_evaluasi_v2,
    index=False,
    encoding="utf-8"
)

print("Hasil evaluasi model Intelligence V2 disimpan:")
print(lokasi_hasil_evaluasi_v2)

data_hasil_evaluasi_v2

Evaluasi Random Forest Intelligence V2
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     26980
    Phishing       1.00      1.00      1.00     20238

    accuracy                           1.00     47218
   macro avg       1.00      1.00      1.00     47218
weighted avg       1.00      1.00      1.00     47218

Confusion Matrix:
[[26964    16]
 [   92 20146]]
Evaluasi XGBoost Intelligence V2
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     26980
    Phishing       1.00      0.99      1.00     20238

    accuracy                           1.00     47218
   macro avg       1.00      1.00      1.00     47218
weighted avg       1.00      1.00      1.00     47218

Confusion Matrix:
[[26974     6]
 [  105 20133]]
Hasil evaluasi model Intelligence V2 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_evaluasi_model_intelligence_v2.csv


,nama_model,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc
0,Random Forest Intelligence V2,0.9977,0.9992,0.9955,0.9973,0.9990
1,XGBoost Intelligence V2,0.9976,0.9997,0.9948,0.9973,0.9991


## Uji Kasus Khusus Website Resmi dan Tiruan

In [15]:
def buat_fitur_v2_dari_daftar_url(daftar_url):
    X_manual = buat_fitur_manual_dari_daftar_url(
        daftar_url,
        daftar_fitur_url_manual
    )

    data_intelligence = url_intelligence.analisis_banyak_url(
        daftar_url,
        data_domain_resmi,
        data_brand_keyword,
        data_suspicious_keyword
    )

    X_intelligence = data_intelligence[FITUR_INTELLIGENCE_V2].copy()
    X_intelligence = X_intelligence.fillna(0)

    X_v2 = pd.concat(
        [
            X_manual.reset_index(drop=True),
            X_intelligence.reset_index(drop=True)
        ],
        axis=1
    )

    X_v2 = X_v2.reindex(columns=daftar_fitur_model_v2, fill_value=0)

    return X_v2, data_intelligence


daftar_url_uji_khusus = [
    "https://praktikum.gunadarma.ac.id",
    "https://baak.gunadarma.ac.id",
    "https://www.bca.co.id",
    "https://www.shopee.co.id",
    "https://www.microsoft.com",
    "http://rricrosoft.com",
    "http://rnicrosoft.com",
    "http://micros0ft-login-update.test",
    "http://bca-login-update.test",
    "http://paypal-verify-account.test",
    "http://praktikum-gunadarma-login-update.test",
    "https://xn--micrsoft-q4a.test"
]

X_uji_khusus_v2, data_intelligence_uji_khusus = buat_fitur_v2_dari_daftar_url(
    daftar_url_uji_khusus
)

probabilitas_rf = model_rf_intelligence_v2.predict_proba(X_uji_khusus_v2)[:, 1]
prediksi_rf = model_rf_intelligence_v2.predict(X_uji_khusus_v2)

data_uji_khusus_v2 = pd.DataFrame({
    "url": daftar_url_uji_khusus,
    "prediksi_rf_v2": prediksi_rf,
    "probabilitas_phishing_rf_v2": probabilitas_rf,
    "skor_risiko_rf_v2": np.round(probabilitas_rf * 100, 2)
})

data_uji_khusus_v2["hasil_rf_v2"] = data_uji_khusus_v2["prediksi_rf_v2"].map({
    0: "Legitimate",
    1: "Phishing"
})

data_uji_khusus_v2 = pd.concat(
    [
        data_uji_khusus_v2,
        data_intelligence_uji_khusus[[
            "is_official_domain",
            "official_brand",
            "brand_detected",
            "brand_but_not_official",
            "suspicious_keywords",
            "suspicious_keyword_score",
            "lookalike_brand_detected",
            "lookalike_brand",
            "lookalike_score",
            "uses_punycode",
            "uses_digit_substitution",
            "intelligence_status",
            "intelligence_reason"
        ]].reset_index(drop=True)
    ],
    axis=1
)

lokasi_uji_khusus_v2 = direktori_outputs / "uji_khusus_model_intelligence_v2.csv"

data_uji_khusus_v2.to_csv(
    lokasi_uji_khusus_v2,
    index=False,
    encoding="utf-8"
)

print("Hasil uji khusus model Intelligence V2 disimpan:")
print(lokasi_uji_khusus_v2)

data_uji_khusus_v2

Hasil uji khusus model Intelligence V2 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\uji_khusus_model_intelligence_v2.csv


,url,prediksi_rf_v2,probabilitas_phishing_rf_v2,skor_risiko_rf_v2,hasil_rf_v2,is_official_domain,official_brand,brand_detected,brand_but_not_official,suspicious_keywords,suspicious_keyword_score,lookalike_brand_detected,lookalike_brand,lookalike_score,uses_punycode,uses_digit_substitution,intelligence_status,intelligence_reason
0,https://praktikum.gunadarma.ac.id,0,0.204000,20.40,Legitimate,1,Gunadarma,Gunadarma,0,,0,0,,0.0000,0,0,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.
1,https://baak.gunadarma.ac.id,0,0.316000,31.60,Legitimate,1,Gunadarma,Gunadarma,0,,0,0,,0.0000,0,0,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.
2,https://www.bca.co.id,0,0.040543,4.05,Legitimate,1,BCA,BCA,0,,0,0,,0.0000,0,0,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.
3,https://www.shopee.co.id,0,0.092183,9.22,Legitimate,1,Shopee,Shopee,0,,0,0,,0.0000,0,0,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.
4,https://www.microsoft.com,0,0.480013,48.00,Legitimate,1,Microsoft,Microsoft,0,,0,0,,0.0000,0,0,resmi_terlihat_aman,Domain cocok dengan daftar domain resmi.
5,http://rricrosoft.com,1,0.996000,99.60,Phishing,0,,,0,,0,1,Microsoft,0.8421,0,0,domain_mirip_brand,Domain terlihat mirip dengan brand resmi.
6,http://rnicrosoft.com,1,0.996000,99.60,Phishing,0,,,0,,0,1,Microsoft,0.8421,0,0,domain_mirip_brand,Domain terlihat mirip dengan brand resmi.
7,http://micros0ft-login-update.test,1,0.988000,98.80,Phishing,0,,Microsoft,1,"login, update",6,1,Microsoft,1.0000,0,1,tiruan_brand_berisiko,"URL mengandung nama brand, tetapi tidak berada..."
8,http://bca-login-update.test,1,0.996000,99.60,Phishing,0,,BCA,1,"login, update",6,1,BCA,1.0000,0,0,tiruan_brand_berisiko,"URL mengandung nama brand, tetapi tidak berada..."
9,http://paypal-verify-account.test,1,1.000000,100.00,Phishing,0,,PayPal,1,"account, verify",5,1,PayPal,1.0000,0,0,tiruan_brand_berisiko,"URL mengandung nama brand, tetapi tidak berada..."


## Feature Importance Model V2

In [16]:
data_importance_rf_v2 = pd.DataFrame({
    "fitur": daftar_fitur_model_v2,
    "importance": model_rf_intelligence_v2.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

lokasi_importance_rf_v2 = direktori_outputs / "feature_importance_rf_intelligence_v2.csv"

data_importance_rf_v2.to_csv(
    lokasi_importance_rf_v2,
    index=False,
    encoding="utf-8"
)

print("Feature importance RF Intelligence V2 disimpan:")
print(lokasi_importance_rf_v2)

data_importance_rf_v2.head(30)

Feature importance RF Intelligence V2 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\feature_importance_rf_intelligence_v2.csv


,fitur,importance
0,IsHTTPS,0.369853
1,NoOfOtherSpecialCharsInURL,0.115341
2,SpacialCharRatioInURL,0.077243
3,LetterRatioInURL,0.076244
4,DegitRatioInURL,0.064321
5,NoOfDegitsInURL,0.057520
6,NoOfLettersInURL,0.049120
7,URLLength,0.044508
8,NoOfSubDomain,0.037734
9,DomainLength,0.023871


## Simpan Model Intelligence V2

In [17]:
lokasi_model_rf_intelligence_v2 = direktori_models / "model_rf_intelligence_v2.pkl"
lokasi_model_xgb_intelligence_v2 = direktori_models / "model_xgb_intelligence_v2.pkl"
lokasi_model_terbaik_intelligence_v2 = direktori_models / "model_terbaik_intelligence_v2.pkl"

joblib.dump(model_rf_intelligence_v2, lokasi_model_rf_intelligence_v2)

if model_xgb_intelligence_v2 is not None:
    joblib.dump(model_xgb_intelligence_v2, lokasi_model_xgb_intelligence_v2)

nama_model_terbaik = data_hasil_evaluasi_v2.iloc[0]["nama_model"]

if nama_model_terbaik == "XGBoost Intelligence V2" and model_xgb_intelligence_v2 is not None:
    model_terbaik_intelligence_v2 = model_xgb_intelligence_v2
else:
    model_terbaik_intelligence_v2 = model_rf_intelligence_v2
    nama_model_terbaik = "Random Forest Intelligence V2"

joblib.dump(model_terbaik_intelligence_v2, lokasi_model_terbaik_intelligence_v2)

print("Model RF Intelligence V2 disimpan:")
print(lokasi_model_rf_intelligence_v2)

if model_xgb_intelligence_v2 is not None:
    print("Model XGBoost Intelligence V2 disimpan:")
    print(lokasi_model_xgb_intelligence_v2)

print("Model terbaik Intelligence V2 disimpan:")
print(lokasi_model_terbaik_intelligence_v2)
print("Nama model terbaik:", nama_model_terbaik)

Model RF Intelligence V2 disimpan:
C:\Users\ASUS\PHISHING\models\model_rf_intelligence_v2.pkl
Model XGBoost Intelligence V2 disimpan:
C:\Users\ASUS\PHISHING\models\model_xgb_intelligence_v2.pkl
Model terbaik Intelligence V2 disimpan:
C:\Users\ASUS\PHISHING\models\model_terbaik_intelligence_v2.pkl
Nama model terbaik: Random Forest Intelligence V2


## Simpan Metadata Model Intelligence V2

In [18]:
metadata_model_intelligence_v2 = {
    "nama_project": "PhishRisk Intelligence System",
    "step": "STEP 8",
    "nama_model": nama_model_terbaik,
    "versi_model": "Intelligence V2",
    "jumlah_data_training": int(len(data_training_v2)),
    "jumlah_fitur_url_manual": int(len(daftar_fitur_url_manual)),
    "jumlah_fitur_intelligence": int(len(FITUR_INTELLIGENCE_V2)),
    "jumlah_fitur_total": int(len(daftar_fitur_model_v2)),
    "fitur_intelligence": FITUR_INTELLIGENCE_V2,
    "file_dataset_training_v2": str(lokasi_training_v2),
    "file_daftar_fitur_v2": str(lokasi_fitur_intelligence_v2),
    "file_model_rf_v2": str(lokasi_model_rf_intelligence_v2),
    "file_model_terbaik_v2": str(lokasi_model_terbaik_intelligence_v2),
    "file_evaluasi_v2": str(lokasi_hasil_evaluasi_v2),
    "file_uji_khusus_v2": str(lokasi_uji_khusus_v2),
    "file_feature_importance_v2": str(lokasi_importance_rf_v2),
    "catatan": "Model V2 dilatih dengan fitur URL manual dan fitur intelligence tambahan."
}

if model_xgb_intelligence_v2 is not None:
    metadata_model_intelligence_v2["file_model_xgb_v2"] = str(lokasi_model_xgb_intelligence_v2)

lokasi_metadata_model_intelligence_v2 = direktori_outputs / "metadata_model_intelligence_v2.json"

with open(lokasi_metadata_model_intelligence_v2, "w", encoding="utf-8") as file:
    json.dump(metadata_model_intelligence_v2, file, indent=4, ensure_ascii=False)

print("Metadata model Intelligence V2 disimpan:")
print(lokasi_metadata_model_intelligence_v2)

metadata_model_intelligence_v2

Metadata model Intelligence V2 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\metadata_model_intelligence_v2.json


{'nama_project': 'PhishRisk Intelligence System',
 'step': 'STEP 8',
 'nama_model': 'Random Forest Intelligence V2',
 'versi_model': 'Intelligence V2',
 'jumlah_data_training': 236087,
 'jumlah_fitur_url_manual': 39,
 'jumlah_fitur_intelligence': 10,
 'jumlah_fitur_total': 49,
 'fitur_intelligence': ['is_official_domain',
  'brand_keyword_detected',
  'brand_but_not_official',
  'suspicious_keyword_count',
  'suspicious_keyword_score',
  'lookalike_brand_detected',
  'lookalike_score',
  'uses_punycode',
  'uses_digit_substitution',
  'hyphen_count'],
 'file_dataset_training_v2': 'C:\\Users\\ASUS\\PHISHING\\data\\processed\\dataset_training_intelligence_v2.csv',
 'file_daftar_fitur_v2': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\daftar_fitur_intelligence_v2.json',
 'file_model_rf_v2': 'C:\\Users\\ASUS\\PHISHING\\models\\model_rf_intelligence_v2.pkl',
 'file_model_terbaik_v2': 'C:\\Users\\ASUS\\PHISHING\\models\\model_terbaik_intelligence_v2.pkl',
 'file_evaluasi_v2': 'C:\\Users\\ASU

## Validasi File

In [19]:
daftar_file_validasi_step8 = [
    lokasi_training_v2,
    lokasi_fitur_intelligence_v2,
    lokasi_hasil_evaluasi_v2,
    lokasi_uji_khusus_v2,
    lokasi_importance_rf_v2,
    lokasi_model_rf_intelligence_v2,
    lokasi_model_terbaik_intelligence_v2,
    lokasi_metadata_model_intelligence_v2
]

if model_xgb_intelligence_v2 is not None:
    daftar_file_validasi_step8.append(lokasi_model_xgb_intelligence_v2)

hasil_validasi_step8 = []

for lokasi_file in daftar_file_validasi_step8:
    hasil_validasi_step8.append({
        "nama_file": lokasi_file.name,
        "lokasi": str(lokasi_file),
        "tersedia": lokasi_file.exists(),
        "ukuran_kb": round(lokasi_file.stat().st_size / 1024, 2) if lokasi_file.exists() else 0
    })

dataframe_validasi_step8 = pd.DataFrame(hasil_validasi_step8)

lokasi_validasi_step8 = direktori_outputs / "validasi_step8_model_intelligence_v2.csv"

dataframe_validasi_step8.to_csv(
    lokasi_validasi_step8,
    index=False,
    encoding="utf-8"
)

print("Validasi STEP 8 disimpan:")
print(lokasi_validasi_step8)

dataframe_validasi_step8

Validasi STEP 8 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\validasi_step8_model_intelligence_v2.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,dataset_training_intelligence_v2.csv,C:\Users\ASUS\PHISHING\data\processed\dataset_...,True,39484.79
1,daftar_fitur_intelligence_v2.json,C:\Users\ASUS\PHISHING\reports\outputs\daftar_...,True,1.02
2,hasil_evaluasi_model_intelligence_v2.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_e...,True,0.20
3,uji_khusus_model_intelligence_v2.csv,C:\Users\ASUS\PHISHING\reports\outputs\uji_khu...,True,3.01
4,feature_importance_rf_intelligence_v2.csv,C:\Users\ASUS\PHISHING\reports\outputs\feature...,True,1.73
5,model_rf_intelligence_v2.pkl,C:\Users\ASUS\PHISHING\models\model_rf_intelli...,True,49114.43
6,model_terbaik_intelligence_v2.pkl,C:\Users\ASUS\PHISHING\models\model_terbaik_in...,True,49114.43
7,metadata_model_intelligence_v2.json,C:\Users\ASUS\PHISHING\reports\outputs\metadat...,True,1.60
8,model_xgb_intelligence_v2.pkl,C:\Users\ASUS\PHISHING\models\model_xgb_intell...,True,491.41


## Ringkasan

In [20]:
print("RINGKASAN STEP INI")
print("=" * 60)

print("Nama model terbaik:", nama_model_terbaik)
print("Jumlah data training V2:", len(data_training_v2))
print("Jumlah fitur total:", len(daftar_fitur_model_v2))

print("\nDistribusi sumber data:")
print(data_training_v2["sumber_data"].value_counts())

print("\nDistribusi target:")
print(data_training_v2["target_phishing"].value_counts())

print("\nHasil evaluasi model:")
display(data_hasil_evaluasi_v2)

print("\nHasil uji khusus:")
display(data_uji_khusus_v2[[
    "url",
    "hasil_rf_v2",
    "skor_risiko_rf_v2",
    "is_official_domain",
    "brand_detected",
    "brand_but_not_official",
    "suspicious_keywords",
    "lookalike_brand",
    "lookalike_score",
    "intelligence_status"
]])

print("\nFile penting:")
print(lokasi_model_terbaik_intelligence_v2)
print(lokasi_fitur_intelligence_v2)
print(lokasi_metadata_model_intelligence_v2)
print(lokasi_validasi_step8)

RINGKASAN STEP INI
Nama model terbaik: Random Forest Intelligence V2
Jumlah data training V2: 236087
Jumlah fitur total: 49

Distribusi sumber data:
sumber_data
dataset_asli                235795
synthetic_suspicious_url       242
official_domain_seed            50
Name: count, dtype: int64

Distribusi target:
target_phishing
0    134900
1    101187
Name: count, dtype: int64

Hasil evaluasi model:


,nama_model,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc
0,Random Forest Intelligence V2,0.9977,0.9992,0.9955,0.9973,0.9990
1,XGBoost Intelligence V2,0.9976,0.9997,0.9948,0.9973,0.9991



Hasil uji khusus:


,url,hasil_rf_v2,skor_risiko_rf_v2,is_official_domain,brand_detected,brand_but_not_official,suspicious_keywords,lookalike_brand,lookalike_score,intelligence_status
0,https://praktikum.gunadarma.ac.id,Legitimate,20.40,1,Gunadarma,0,,,0.0000,resmi_terlihat_aman
1,https://baak.gunadarma.ac.id,Legitimate,31.60,1,Gunadarma,0,,,0.0000,resmi_terlihat_aman
2,https://www.bca.co.id,Legitimate,4.05,1,BCA,0,,,0.0000,resmi_terlihat_aman
3,https://www.shopee.co.id,Legitimate,9.22,1,Shopee,0,,,0.0000,resmi_terlihat_aman
4,https://www.microsoft.com,Legitimate,48.00,1,Microsoft,0,,,0.0000,resmi_terlihat_aman
5,http://rricrosoft.com,Phishing,99.60,0,,0,,Microsoft,0.8421,domain_mirip_brand
6,http://rnicrosoft.com,Phishing,99.60,0,,0,,Microsoft,0.8421,domain_mirip_brand
7,http://micros0ft-login-update.test,Phishing,98.80,0,Microsoft,1,"login, update",Microsoft,1.0000,tiruan_brand_berisiko
8,http://bca-login-update.test,Phishing,99.60,0,BCA,1,"login, update",BCA,1.0000,tiruan_brand_berisiko
9,http://paypal-verify-account.test,Phishing,100.00,0,PayPal,1,"account, verify",PayPal,1.0000,tiruan_brand_berisiko



File penting:
C:\Users\ASUS\PHISHING\models\model_terbaik_intelligence_v2.pkl
C:\Users\ASUS\PHISHING\reports\outputs\daftar_fitur_intelligence_v2.json
C:\Users\ASUS\PHISHING\reports\outputs\metadata_model_intelligence_v2.json
C:\Users\ASUS\PHISHING\reports\outputs\validasi_step8_model_intelligence_v2.csv
